# Segmenter les clients d'Olist pour cibler les campagnes marketing
## Partie 3 : Fréquence de mise à jour de la segmentation

### Sommaire <a class="anchor" id="sommaire"></a>

* [Introduction](#0)
* [Chargement des données](#1)
* [Calcul des indicateurs à une date donnée](#2)
* [Méthode 1 : en avançant dans le temps](#3)
* [Méthode 2 : en remontant dans le temps](#4)
* [Fréquence de mise à jour recommandée](#5)
* [Conclusion](#6)

## Introduction <a class="anchor" id="0"></a>

Le notebook 2 a retenu un **K-Means à 5 clusters** sur la récence, la fréquence, le montant
et la note moyenne. Mais les clients évoluent : de nouveaux clients arrivent, d'autres
passent une nouvelle commande, et la segmentation **devient peu à peu dépassée**.

Ce dernier notebook cherche **au bout de combien de temps il faut la recalculer**, pour proposer
à Olist un **contrat de maintenance** adapté.

**La méthode :**
1. on entraîne le modèle à une **date de départ** passée, sur les clients connus à cette date ;
2. on avance **semaine par semaine**, en ajoutant les nouvelles commandes ;
3. chaque semaine, on compare les segments donnés par le **modèle de départ** à ceux d'un
   **modèle réentraîné** sur les données à jour, avec l'**ARI** (vu dans le notebook 2) ;
4. quand l'ARI passe **sous 0,8**, les deux segmentations sont trop différentes : il faut
   mettre à jour le modèle.

<a href="#sommaire" rev="footnote">&#8617;</a>

## Chargement des données <a class="anchor" id="1"></a>

On part du fichier `data_commandes.csv`, produit par le notebook 1 : une ligne par commande,
avec sa **date**. C'est elle qui permet de recalculer les indicateurs de chaque client
à n'importe quelle date.

In [1]:
# Manipulation des données
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Modélisation
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

# Mesure du temps d'exécution
from time import time

# Style commun à tous les graphiques
sns.set_theme(style="whitegrid", palette="bright")

# Fichier produit par le notebook 1 : une ligne par commande
# parse_dates convertit directement la colonne au format date
commandes = pd.read_csv('data/data_commandes.csv', parse_dates=['order_purchase_timestamp'])

print(f"{commandes.shape[0]:,} commandes, {commandes['customer_unique_id'].nunique():,} clients")
print(f"Du {commandes['order_purchase_timestamp'].min():%d/%m/%Y} "
      f"au {commandes['order_purchase_timestamp'].max():%d/%m/%Y}")
commandes.head()

98,206 commandes, 94,989 clients
Du 04/09/2016 au 03/09/2018


,customer_unique_id,order_id,order_purchase_timestamp,montant_command,n_produits,review_score
0,861eff4711a542e4b93843c6dd7febb0,00e7ee1b050b8499577073aeb2a297a1,2017-05-16 15:05:35,146.87,1,4.0
1,290c77bc529b7ac935b93aa66c333dc3,29150127e6685892b6eab3eec79f59c7,2018-01-12 20:48:24,335.48,1,5.0
2,060e732b5b29e8181a18229c7b0b2b5e,b2059ed67ce144a36e2aa97d2c9e9ad2,2018-05-19 16:07:45,157.73,1,5.0
3,259dac757896d24d7702b9acbbff3f3c,951670f92359f4fe4a63112aa7306eba,2018-03-13 16:06:38,173.30,1,5.0
4,345ecd01c38d18a9036ed96c73b8d066,6b7d50bd145f6fc7f33cebabd7e49d0f,2018-07-29 09:51:30,252.25,1,5.0


<a href="#sommaire" rev="footnote">&#8617;</a>

## Calcul des indicateurs à une date donnée <a class="anchor" id="2"></a>

Pour simuler l'évolution des clients, il faut pouvoir calculer leurs indicateurs **à n'importe
quelle date**, comme si l'analyse avait été faite ce jour-là. La fonction `indicateurs_a_date` :
1. ne garde que les commandes passées **jusqu'à cette date** ;
2. calcule, pour chaque client, les quatre variables du modèle retenu : récence, fréquence,
   montant et note moyenne.

La récence est calculée, comme dans le notebook 1, par rapport au **lendemain** de la date choisie.

In [2]:
from datetime import timedelta


def indicateurs_a_date(date):
    """Récence, fréquence, montant et note moyenne de chaque client, avec les commandes
    passées jusqu'à la date donnée."""
    passees = commandes[commandes['order_purchase_timestamp'] <= date]
    reference = date + timedelta(days=1)   # lendemain de la date choisie

    clients = passees.groupby('customer_unique_id').agg(
        Recency=('order_purchase_timestamp', lambda d: (reference - d.max()).days),
        Frequency=('order_id', 'nunique'),
        Monetary=('montant_command', 'sum'),
        Review_score=('review_score', 'mean')
    )
    return clients


# Vérification : à la dernière date, on doit retrouver les 94 989 clients du notebook 2
derniere_date = commandes['order_purchase_timestamp'].max()

debut = time()
test = indicateurs_a_date(derniere_date)
print(f"Temps de calcul : {time() - debut:.1f} secondes")
print(f"{test.shape[0]:,} clients")
test.describe().round(2)

Temps de calcul : 96.1 secondes
94,989 clients


,Recency,Frequency,Monetary,Review_score
count,94989.00,94989.00,94989.00,94989.00
mean,243.44,1.03,165.69,4.12
std,152.99,0.21,226.74,1.31
min,1.00,1.00,9.59,1.00
25%,119.00,1.00,63.10,4.00
50%,224.00,1.00,107.90,5.00
75%,352.00,1.00,182.94,5.00
max,729.00,16.00,13664.08,5.00


**À retenir :** à la dernière date, la fonction retrouve exactement les indicateurs des notebooks
1 et 2 (94 989 clients, note moyenne de 4,12, récence médiane de 224 jours) : elle peut donc
servir à calculer les indicateurs des clients à n'importe quelle date passée.

<a href="#sommaire" rev="footnote">&#8617;</a>

## Méthode 1 : en avançant dans le temps <a class="anchor" id="3"></a>

#### Le modèle de départ

On se place au **1er janvier 2018**, comme si la segmentation avait été faite ce jour-là.
Cette date laisse **plus d'un an de commandes** pour entraîner le modèle, et **8 mois** de
nouvelles commandes pour observer comment la segmentation vieillit.

On entraîne à cette date le modèle retenu dans le notebook 2 : un **K-Means à 5 clusters** sur
la récence, la fréquence, le montant et la note moyenne. On garde aussi la **standardisation**
calculée à cette date : c'est elle que le modèle de départ appliquerait aux nouvelles données.

In [3]:
variables = ['Recency', 'Frequency', 'Monetary', 'Review_score']
date_depart = pd.Timestamp('2018-01-01')

# Indicateurs des clients connus au 1er janvier 2018
clients_depart = indicateurs_a_date(date_depart)

# Standardisation et modèle de départ, figés à cette date
scaler_depart = StandardScaler().fit(clients_depart[variables])
kmeans_depart = KMeans(n_clusters=5, n_init=10, random_state=0).fit(
    scaler_depart.transform(clients_depart[variables]))

print(f"Clients connus au {date_depart:%d/%m/%Y} : {clients_depart.shape[0]:,}")

# Taille des 5 clusters du modèle de départ
pd.Series(kmeans_depart.labels_).value_counts(normalize=True).sort_index().mul(100).round(1)

Clients connus au 01/01/2018 : 43,321


0    32.7
1    47.1
2     2.8
3    15.4
4     2.0
dtype: float64

**À retenir :** au 1er janvier 2018, Olist compte **43 321 clients**, moins de la moitié de la base
finale. Le modèle de départ retrouve pourtant les mêmes 5 segments que dans le notebook 2 :
deux grands groupes de petits paniers (47 % et 33 %), les clients mécontents (15 %),
les clients fidèles (3 %) et les gros paniers (2 %).

#### Évolution de l'ARI semaine après semaine

Chaque semaine, à partir du 1er janvier 2018 :
1. on recalcule les indicateurs de **tous les clients connus à cette date** (anciens et nouveaux) ;
2. on les classe avec le **modèle de départ**, sans le réentraîner ;
3. on entraîne un **nouveau modèle** sur ces données à jour ;
4. on compare les deux segmentations avec l'**ARI**.

In [ ]:
dates = pd.date_range(date_depart + pd.Timedelta(weeks=1), commandes['order_purchase_timestamp'].max(), freq='7D')

resultats = []
debut = time()
for semaine, date in enumerate(dates, start=1):
    clients = indicateurs_a_date(date)

    # Segments donnés par le modèle de départ, figé au 1er janvier 2018
    labels_depart = kmeans_depart.predict(scaler_depart.transform(clients[variables]))

    # Segments donnés par un nouveau modèle, entraîné sur les données à jour
    X_nouveau = StandardScaler().fit_transform(clients[variables])
    labels_nouveau = KMeans(n_clusters=5, n_init=10, random_state=0).fit_predict(X_nouveau)

    resultats.append({'Semaine': semaine, 'Date': date, 'Clients': len(clients),
                      'ARI': adjusted_rand_score(labels_depart, labels_nouveau)})
    print(f"Semaine {semaine} ({date:%d/%m/%Y}) : ARI = {resultats[-1]['ARI']:.3f}")

print(f"Temps de calcul : {time() - debut:.0f} secondes")
resultats = pd.DataFrame(resultats)

Semaine 1 (08/01/2018) : ARI = 0.967
Semaine 2 (15/01/2018) : ARI = 0.949
Semaine 3 (22/01/2018) : ARI = 0.934
Semaine 4 (29/01/2018) : ARI = 0.908
Semaine 5 (05/02/2018) : ARI = 0.895
Semaine 6 (12/02/2018) : ARI = 0.864
Semaine 7 (19/02/2018) : ARI = 0.844
Semaine 8 (26/02/2018) : ARI = 0.761
Semaine 9 (05/03/2018) : ARI = 0.742
Semaine 10 (12/03/2018) : ARI = 0.733
Semaine 11 (19/03/2018) : ARI = 0.715
Semaine 12 (26/03/2018) : ARI = 0.702
Semaine 13 (02/04/2018) : ARI = 0.688
Semaine 14 (09/04/2018) : ARI = 0.691
Semaine 15 (16/04/2018) : ARI = 0.676
Semaine 16 (23/04/2018) : ARI = 0.664
Semaine 17 (30/04/2018) : ARI = 0.641
Semaine 18 (07/05/2018) : ARI = 0.595
Semaine 19 (14/05/2018) : ARI = 0.576
Semaine 20 (21/05/2018) : ARI = 0.561
Semaine 21 (28/05/2018) : ARI = 0.584
Semaine 22 (04/06/2018) : ARI = 0.578
Semaine 23 (11/06/2018) : ARI = 0.567
Semaine 24 (18/06/2018) : ARI = 0.540
Semaine 25 (25/06/2018) : ARI = 0.517
Semaine 26 (02/07/2018) : ARI = 0.495
Semaine 27 (09/07/201

In [ ]:
SEUIL = 0.8

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(resultats['Semaine'], resultats['ARI'], marker='o', color='steelblue')
ax.axhline(SEUIL, color='red', linestyle='--', label=f'Seuil de mise à jour (ARI = {SEUIL})')

# Première semaine où l'ARI passe sous le seuil, marquée par une ligne verticale
semaine_seuil = resultats.loc[resultats['ARI'] < SEUIL, 'Semaine'].iloc[0]
ax.axvline(semaine_seuil, color='grey', linestyle=':', label=f'Passage sous le seuil (semaine {semaine_seuil})')

ax.set_title("Évolution de l'ARI entre le modèle de départ et un modèle réentraîné")
ax.set_xlabel('Semaines écoulées depuis le 1er janvier 2018')
ax.set_ylabel('ARI')
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout()
fig.savefig('images/evolution_ari.png', dpi=150, bbox_inches='tight')
plt.show()

premiere = resultats[resultats['ARI'] < SEUIL].iloc[0]
print(f"L'ARI passe sous {SEUIL} à la semaine {premiere['Semaine']} "
      f"({premiere['Date']:%d/%m/%Y}), avec un ARI de {premiere['ARI']:.3f}")

**À retenir :** l'ARI baisse **régulièrement**, d'environ 0,02 par semaine. Il reste au-dessus
du seuil de 0,8 pendant **7 semaines** (0,84 au 19 février 2018), puis passe en dessous à la
**semaine 8** (0,76 au 26 février), avec une chute plus marquée que les semaines précédentes.
Après 8 mois, il n'est plus que d'environ 0,45 : sans mise à jour, la segmentation
devient très différente de la réalité des clients.

<a href="#sommaire" rev="footnote">&#8617;</a>

## Méthode 2 : en remontant dans le temps <a class="anchor" id="4"></a>

On fait le chemin inverse : on entraîne le modèle à la **dernière date** des données
(le 3 septembre 2018), puis on **recule semaine par semaine** jusqu'au 1er janvier 2018.
Chaque semaine, on compare les segments donnés par ce modèle à ceux d'un modèle entraîné
à cette date passée.

Cette méthode répond à la même question de l'autre côté : **jusqu'à quelle date passée la
segmentation actuelle reste-t-elle valable ?** Si les deux méthodes donnent un délai proche,
la recommandation est plus solide.

In [ ]:
date_fin = commandes['order_purchase_timestamp'].max()

# Modèle entraîné à la dernière date, puis figé
clients_fin = indicateurs_a_date(date_fin)
scaler_fin = StandardScaler().fit(clients_fin[variables])
kmeans_fin = KMeans(n_clusters=5, n_init=10, random_state=0).fit(
    scaler_fin.transform(clients_fin[variables]))

# Dates en remontant, semaine par semaine, jusqu'au 1er janvier 2018
dates_retour = pd.date_range(date_fin - pd.Timedelta(weeks=1), date_depart, freq='-7D')

resultats_2 = []
debut = time()
for semaine, date in enumerate(dates_retour, start=1):
    clients = indicateurs_a_date(date)

    # Segments donnés par le modèle de la dernière date
    labels_fin = kmeans_fin.predict(scaler_fin.transform(clients[variables]))

    # Segments donnés par un modèle entraîné à cette date passée
    X_passe = StandardScaler().fit_transform(clients[variables])
    labels_passe = KMeans(n_clusters=5, n_init=10, random_state=0).fit_predict(X_passe)

    resultats_2.append({'Semaine': semaine, 'Date': date, 'Clients': len(clients),
                        'ARI': adjusted_rand_score(labels_fin, labels_passe)})
    print(f"Semaine -{semaine} ({date:%d/%m/%Y}) : ARI = {resultats_2[-1]['ARI']:.3f}")

print(f"Temps de calcul : {time() - debut:.0f} secondes")
resultats_2 = pd.DataFrame(resultats_2)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(resultats_2['Semaine'], resultats_2['ARI'], marker='o', color='steelblue')
ax.axhline(SEUIL, color='red', linestyle='--', label=f'Seuil de mise à jour (ARI = {SEUIL})')

# Première semaine où l'ARI passe sous le seuil, marquée par une ligne verticale
semaine_seuil_2 = resultats_2.loc[resultats_2['ARI'] < SEUIL, 'Semaine'].iloc[0]
ax.axvline(semaine_seuil_2, color='grey', linestyle=':',
           label=f'Passage sous le seuil (semaine -{semaine_seuil_2})')

ax.set_title("Méthode 2 : évolution de l'ARI en remontant dans le temps")
ax.set_xlabel('Semaines avant le 3 septembre 2018')
ax.set_ylabel('ARI')
ax.set_ylim(0, 1.05)
ax.legend()
plt.tight_layout()
fig.savefig('images/evolution_ari_methode2.png', dpi=150, bbox_inches='tight')
plt.show()

premiere_2 = resultats_2[resultats_2['ARI'] < SEUIL].iloc[0]
print(f"L'ARI passe sous {SEUIL} à la semaine -{premiere_2['Semaine']} "
      f"({premiere_2['Date']:%d/%m/%Y}), avec un ARI de {premiere_2['ARI']:.3f}")

**À retenir :** en remontant depuis le 3 septembre 2018, l'ARI reste au-dessus de 0,8 pendant
**13 semaines** (0,80 au 4 juin 2018), puis passe en dessous à la **semaine -14** (0,78 au 28 mai).
C'est presque **deux fois plus long** qu'avec la méthode 1 (7 semaines). Cette méthode ne
reproduit pas l'usage réel : en pratique, Olist entraînera un modèle à une date donnée, puis
l'appliquera aux clients des semaines **suivantes**, pas des semaines passées. La vérification
ci-dessous, qui avance dans le temps sur la période la plus récente, permet de trancher.

#### Vérification sur la période la plus récente

Les deux méthodes ne donnent pas le même délai : 7 semaines en partant de janvier 2018,
13 semaines en remontant depuis septembre. Pour trancher, on refait la méthode 1 sur la période
la plus récente : on fige un modèle **13 semaines avant la fin des données** (le 4 juin 2018),
puis on avance semaine par semaine jusqu'au 3 septembre.

Si l'ARI reste au-dessus de 0,8 pendant ces 13 semaines, la segmentation **vieillit moins vite
quand la base de clients est plus grande**, et le délai de la méthode 2 est confirmé. Sinon,
c'est le délai plus court de la méthode 1 qu'il faut retenir.

In [ ]:
PERIODE = 13

date_debut_verif = date_fin - pd.Timedelta(weeks=PERIODE)

# Modèle entraîné au début de la période vérifiée, puis figé
clients_verif = indicateurs_a_date(date_debut_verif)
scaler_verif = StandardScaler().fit(clients_verif[variables])
kmeans_verif = KMeans(n_clusters=5, n_init=10, random_state=0).fit(
    scaler_verif.transform(clients_verif[variables]))

resultats_verif = []
for semaine in range(1, PERIODE + 1):
    date = date_debut_verif + pd.Timedelta(weeks=semaine)
    clients = indicateurs_a_date(date)

    labels_verif = kmeans_verif.predict(scaler_verif.transform(clients[variables]))
    labels_nouveau = KMeans(n_clusters=5, n_init=10, random_state=0).fit_predict(
        StandardScaler().fit_transform(clients[variables]))

    resultats_verif.append({'Semaine': semaine, 'Date': date,
                            'ARI': adjusted_rand_score(labels_verif, labels_nouveau)})
    print(f"Semaine {semaine} ({date:%d/%m/%Y}) : ARI = {resultats_verif[-1]['ARI']:.3f}")

resultats_verif = pd.DataFrame(resultats_verif)
print(f"\nARI minimum sur les {PERIODE} semaines : {resultats_verif['ARI'].min():.3f}")

**À retenir :** sur la période la plus récente, l'ARI reste au-dessus de 0,8 pendant
**6 semaines** (0,81 au 16 juillet 2018), puis passe en dessous à la **semaine 7** (0,78 au
23 juillet). La segmentation vieillit donc **aussi vite qu'en début d'année** (7 semaines avec
la méthode 1). Les deux simulations qui avancent dans le temps, comme le ferait Olist en pratique,
donnent un délai de **6 à 7 semaines** : c'est ce délai qu'il faut retenir.

<a href="#sommaire" rev="footnote">&#8617;</a>

## Fréquence de mise à jour recommandée <a class="anchor" id="5"></a>

| Simulation | ARI au-dessus de 0,8 pendant |
|---|---|
| Méthode 1 : en avançant depuis le 1er janvier 2018 | **7 semaines** |
| Méthode 2 : en remontant depuis le 3 septembre 2018 | 13 semaines |
| Vérification : en avançant depuis le 4 juin 2018 | **6 semaines** |

Les deux simulations qui **avancent dans le temps**, comme le ferait Olist en pratique, donnent
un délai proche : **6 à 7 semaines**, au début comme à la fin de la période. La méthode 2 donne
un délai plus long, mais elle applique un modèle récent à des clients du passé, ce qui ne
correspond pas à l'usage réel.

**Recommandation :** réentraîner la segmentation **toutes les 6 semaines**. C'est le délai le
plus court observé : il garantit que la segmentation reste fidèle aux clients (ARI au-dessus
de 0,8) dans toutes les simulations réalistes.

<a href="#sommaire" rev="footnote">&#8617;</a>

## Conclusion <a class="anchor" id="6"></a>

**Ce qui a été fait :** on a mesuré, semaine après semaine, à quel point la segmentation
du notebook 2 **vieillit**, en comparant un modèle figé à un modèle réentraîné sur les données
à jour, avec l'ARI. Deux méthodes ont été comparées, puis le résultat a été vérifié sur
la période la plus récente.

**Recommandation pour Olist :** réentraîner la segmentation **toutes les 6 semaines**, soit
environ **tous les mois et demi**. C'est la durée à prévoir dans le contrat de maintenance.

**Pourquoi la segmentation vieillit-elle ?** Chaque semaine, la **récence** de tous les clients
augmente de 7 jours, et de **nouveaux clients** arrivent. Un modèle figé ne tient compte
ni de l'un ni de l'autre.

**Limites :**
- le seuil de 0,8 est une **convention** : un seuil plus strict conduirait à des mises à jour
  plus fréquentes ;
- la simulation ne couvre que **8 mois** : il faudrait la refaire avec plus de recul pour tenir
  compte des effets saisonniers, comme le Black Friday ;
- la méthode 2 montre que le **choix de la simulation** influence fortement le résultat :
  c'est pourquoi la recommandation s'appuie sur les simulations les plus proches de l'usage réel.

<a href="#sommaire" rev="footnote">&#8617;</a>